# Démo 2 : régularisation, scheduler et clipping

Objectif : voir le dropout changer de comportement entre `train()` et `eval()`, comparer un CNN sans et avec dropout et batch norm, comparer un learning rate constant, StepLR et CosineAnnealingLR sur le même CNN, et mesurer la norme des gradients avant et après clipping.

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- le dropout, qui ne fait pas la même chose en entraînement et en évaluation ;
- l'effet du dropout et de la batch norm sur un CNN qui surapprend ;
- deux outils pour piloter l'optimisation : le scheduler de learning rate, comparé à un learning rate constant sur le même CNN, et le gradient clipping.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
torchviz==0.0.3
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

**On télécharge Fashion-MNIST une fois pour toutes dans le dossier `data`.**

Le code :
- `datasets.FashionMNIST("data", train=..., download=True)` : télécharge le jeu d'entraînement, puis celui de
  test, s'ils ne sont pas déjà là ;
- `%%capture` : masque la barre de progression du téléchargement.

In [ ]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Des données réduites

**On n'entraîne que sur 1000 images, pour que le surapprentissage apparaisse vite.**

Avec peu d'images, un réseau finit par les apprendre par coeur. Il réussit alors très bien sur ses
images d'entraînement et moins bien sur des images nouvelles : c'est le **surapprentissage** (overfitting).
C'est la situation d'un projet où les données étiquetées sont rares.

Le code :
- `torch.randperm(len(complet))[:1000]` : 1000 numéros d'images tirés au hasard ;
- `Subset(complet, ...)` : le Dataset réduit à ces images ;
- `Subset(test, range(2000))` : 2000 images de test serviront de validation.

In [ ]:
import torch
from torch.utils.data import Subset
from torchvision import datasets, transforms

complet = datasets.FashionMNIST("data", train=True, transform=transforms.ToTensor())
test = datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())
torch.manual_seed(0)
train_ds = Subset(complet, torch.randperm(len(complet))[:1000].tolist())
val_ds = Subset(test, range(2000))

Rien à afficher : 1000 images d'entraînement et 2000 de validation sont prêtes.

## 3. Le dropout en train() et en eval()

**Le dropout éteint au hasard une partie des neurones, pendant l'entraînement seulement.**

Si des neurones peuvent disparaître à tout moment, le réseau ne peut pas compter sur quelques-uns
pour apprendre par coeur ; il doit répartir l'information. En évaluation, on veut au contraire le réseau
complet.

Le code :
- `nn.Dropout(p=0.5)` : chaque valeur est mise à zéro avec une probabilité de 0.5 ; les autres sont
  multipliées par 2, pour garder la même moyenne ;
- `dropout.train()`, puis `dropout.eval()` : les deux modes, sur le même vecteur de 1.

In [7]:
import torch.nn as nn

torch.manual_seed(0)
dropout = nn.Dropout(p=0.5)
dropout.train()
print("train :", dropout(torch.ones(10)))
dropout.eval()
print("eval  :", dropout(torch.ones(10)))

En mode `train()`, le dropout a mis 6 valeurs sur 10 à zéro et multiplié les 4 autres par 2. En mode `eval()`, il laisse passer le vecteur intact. D'où l'importance de `model.eval()` avant toute évaluation.

**Piège fréquent** : oublier `model.eval()` avant de mesurer ; le dropout reste actif et les mesures changent d'une exécution à l'autre.

## 4. Un CNN, avec ou sans régularisation (fourni)

**Une classe qui construit le même CNN, avec ou sans régularisation.**

Pour mesurer l'effet d'une technique, on compare le même modèle avec et sans elle, toutes choses
égales par ailleurs.

Le code (à survoler) :
- `features` : deux blocs convolution (32 puis 64 canaux), ReLU, pooling ; `classifieur` : la partie
  linéaire ; `forward` enchaîne les deux ;
- `batchnorm=True` : ajoute `nn.BatchNorm2d` après chaque convolution ; la **batch norm** normalise les
  activations de chaque lot, ce qui stabilise l'entraînement ;
- `dropout=0.25` : ajoute `nn.Dropout2d`, qui éteint des cartes entières, et un `nn.Dropout` avant la
  dernière couche.

In [ ]:
class CNN(nn.Module):
    def __init__(self, batchnorm=False, dropout=0.0):
        super().__init__()
        couches = []
        for entree, sortie in [(1, 32), (32, 64)]:
            couches.append(nn.Conv2d(entree, sortie, kernel_size=3, padding=1))
            if batchnorm:
                couches.append(nn.BatchNorm2d(sortie))
            couches += [nn.ReLU(), nn.MaxPool2d(2)]
            if dropout > 0:
                couches.append(nn.Dropout2d(p=dropout))
        self.features = nn.Sequential(*couches)
        classifieur = [nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU()]
        if dropout > 0:
            classifieur.append(nn.Dropout(p=0.5))
        classifieur.append(nn.Linear(128, 10))
        self.classifieur = nn.Sequential(*classifieur)

    def forward(self, x):
        return self.classifieur(self.features(x))

Rien à afficher : la classe est définie ; elle construira trois versions du même CNN.

## 5. Mesurer accuracy et loss (fourni)

**Une fonction qui mesure l'accuracy et la loss d'un modèle sur un jeu entier.**

Le code (à survoler) :
- `model.eval()` et `torch.no_grad()` : on mesure sans entraîner ;
- tout le jeu en un seul lot (`batch_size=len(ds)`), puisqu'il est petit ;
- renvoie l'accuracy et la loss (`cross_entropy`).

In [ ]:
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def mesurer(model, ds):
    model.eval()
    images, labels = next(iter(DataLoader(ds, batch_size=len(ds))))
    with torch.no_grad():
        scores = model(images.to(device)).cpu()
    return (scores.argmax(1) == labels).float().mean().item(), nn.functional.cross_entropy(scores, labels).item()

Rien à afficher : la fonction renvoie l'accuracy et la loss sur un jeu de données.

## 6. Entraîner 30 époques (fourni)

**La boucle habituelle, 30 époques sur les 1000 images.**

Le code (à survoler) : `Adam`, des lots de 32 images mélangées, puis pour chaque lot `zero_grad()`,
`backward()`, `step()`. `torch.manual_seed(0)` garantit que les trois versions voient les images dans le même
ordre. Après chaque époque, `mesurer` donne l'accuracy et la loss sur l'entraînement et la validation, et
`writer.add_scalars(...)` les écrit pour TensorBoard, sous `runs/regularisation/<nom>`.

In [ ]:
from torch.utils.tensorboard import SummaryWriter


def entrainer(model, nom):
    torch.manual_seed(0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    writer = SummaryWriter(f"runs/regularisation/{nom}")
    for epoch in range(1, 31):
        model.train()
        for images, labels in DataLoader(train_ds, batch_size=32, shuffle=True):
            loss = nn.functional.cross_entropy(model(images.to(device)), labels.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        (train_acc, train_loss), (val_acc, val_loss) = mesurer(model, train_ds), mesurer(model, val_ds)
        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
    writer.close()

Rien à afficher : la fonction est définie.

## 7. Comparer trois versions

**Trois versions du même CNN : sans régularisation, avec dropout, avec dropout et batch norm.**

On lit deux indicateurs. L'**écart** entre l'accuracy d'entraînement et celle de validation mesure
le surapprentissage ; la **loss de validation** mesure la qualité sur des images nouvelles.

Le code :
- la boucle crée chaque version avec `CNN(**options)`, l'entraîne, puis la mesure sur l'entraînement et sur la
  validation ;
- `écart` : accuracy d'entraînement moins accuracy de validation.

Prédisez : quelle version aura le plus grand écart ?

In [12]:
for nom, options in [("sans régularisation", {}), ("dropout", dict(dropout=0.25)),
                     ("dropout + batch norm", dict(batchnorm=True, dropout=0.25))]:
    torch.manual_seed(0)
    model = CNN(**options).to(device)
    entrainer(model, nom)
    (acc_trn, _), (acc_val, loss_val) = mesurer(model, train_ds), mesurer(model, val_ds)
    print(f"{nom:21s} train_acc {acc_trn:.3f} val_acc {acc_val:.3f}  écart {acc_trn - acc_val:.3f}  val_loss {loss_val:.3f}")

Sans régularisation, le modèle apprend presque par cœur ses images d'entraînement (0.974) pour 0.830 en validation : écart de 0.144, loss de validation de 0.701. Le dropout réduit nettement l'écart (0.089) et fait baisser la loss de validation (0.524). Dropout et batch norm donnent la meilleure validation (0.845) et la loss la plus basse (0.502), avec un écart intermédiaire (0.117).

**À retenir** : on juge une régularisation sur la validation, pas sur l'entraînement.

## 8. Les trois versions dans TensorBoard

**TensorBoard affiche les courbes écrites pendant l'entraînement, comme dans la démo 5.**

Le code :
- `%load_ext tensorboard` : charge l'extension dans le notebook ;
- `%tensorboard --logdir runs/regularisation` : affiche les courbes de ce dossier ; chaque run a sa couleur, et on peut
  les cocher ou les décocher pour les comparer.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/regularisation

Dans Colab, le tableau de bord de TensorBoard s'affiche sous la cellule ; ce notebook enregistré n'en garde pas l'image. Les trois versions y sont superposées : on suit époque par époque l'écart entre entraînement et validation que résument les chiffres de l'étape précédente.

## 9. Le learning rate avec StepLR

**Un scheduler fait varier le learning rate au fil des époques.**

Au début, de grands pas font progresser vite ; à la fin, de petits pas affinent sans tout dérégler.
StepLR divise le learning rate par un facteur toutes les N époques.

Le code :
- `StepLR(optimizer, step_size=10, gamma=0.1)` : toutes les 10 époques, le learning rate est multiplié par 0.1 ;
- `scheduler.get_last_lr()` : la valeur actuelle ;
- `scheduler.step()` : une fois par époque, après `optimizer.step()`. Ici, on ne fait que relever les valeurs,
  sans entraîner.

In [14]:
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR

optimizer = torch.optim.SGD(nn.Linear(2, 1).parameters(), lr=0.05)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
lr_step = []
for epoch in range(30):
    lr_step.append(scheduler.get_last_lr()[0])
    optimizer.step()
    scheduler.step()

print("StepLR :", lr_step)

Rien à afficher : les 30 valeurs du learning rate sont relevées dans `lr_step`.

## 10. Le learning rate avec CosineAnnealingLR

**Un autre scheduler, qui fait décroître le learning rate en douceur.**

Le code :
- `CosineAnnealingLR(optimizer, T_max=30)` : le learning rate suit une demi-courbe en cosinus, de sa valeur de
  départ jusqu'à presque zéro en 30 époques ;
- le relevé est le même qu'à l'étape précédente.

In [ ]:
optimizer = torch.optim.SGD(nn.Linear(2, 1).parameters(), lr=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=30)
lr_cos = []
for epoch in range(30):
    lr_cos.append(scheduler.get_last_lr()[0])
    optimizer.step()
    scheduler.step()

Rien à afficher : les 30 valeurs sont relevées dans `lr_cos`.

## 11. Comparer les deux courbes

**Les deux learning rates, époque par époque.**

Prédisez : quelle courbe sera en escalier, laquelle sera lisse ?

In [ ]:
import matplotlib.pyplot as plt

plt.plot(lr_step, label="StepLR")
plt.plot(lr_cos, label="CosineAnnealingLR")
plt.xlabel("époque")
plt.ylabel("learning rate")
plt.legend()
plt.show()

StepLR descend en paliers : le learning rate est divisé par dix toutes les dix époques. CosineAnnealingLR le fait décroître en douceur, jusqu'à presque zéro à la dernière époque.

**À retenir** : `scheduler.step()` s'appelle une fois par époque, après les `optimizer.step()` de l'époque.

## 12. Entraîner avec un scheduler (fourni)

**La boucle de l'étape 6, avec un scheduler en plus.**

On garde tout le reste identique (même modèle, même optimiseur, mêmes images dans le même ordre) pour que
seul le learning rate change d'un entraînement à l'autre.

Le code (à survoler) :
- `nom` choisit le scheduler : `StepLR`, `CosineAnnealingLR`, ou aucun (`scheduler = None`, learning rate
  constant) ;
- `scheduler.step()` : une fois par époque, après tous les `optimizer.step()` de l'époque ;
- `optimizer.param_groups[0]["lr"]` : le learning rate utilisé pendant l'époque, écrit pour TensorBoard avec
  `writer.add_scalar` ;
- une ligne par époque : loss et accuracy sur l'entraînement, puis sur la validation.

In [ ]:
def entrainer_avec_scheduler(model, nom):
    torch.manual_seed(0)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    if nom == "StepLR":
        scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
    elif nom == "CosineAnnealingLR":
        scheduler = CosineAnnealingLR(optimizer, T_max=30)
    else:
        scheduler = None
    writer = SummaryWriter(f"runs/scheduler/{nom}")
    for epoch in range(1, 31):
        writer.add_scalar("learning rate", optimizer.param_groups[0]["lr"], epoch)
        model.train()
        for images, labels in DataLoader(train_ds, batch_size=32, shuffle=True):
            loss = nn.functional.cross_entropy(model(images.to(device)), labels.to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        (train_acc, train_loss), (val_acc, val_loss) = mesurer(model, train_ds), mesurer(model, val_ds)
        print(f"époque {epoch} : train_loss {train_loss:.3f}  train_acc {train_acc:.3f}  val_loss {val_loss:.3f}  val_acc {val_acc:.3f}")
        writer.add_scalars("loss", {"train": train_loss, "val": val_loss}, epoch)
        writer.add_scalars("accuracy", {"train": train_acc, "val": val_acc}, epoch)
    writer.close()

Rien à afficher : la fonction est définie.

## 13. Comparer trois learning rates

**Trois entraînements du même CNN, qui ne diffèrent que par le learning rate.**

Le modèle est la version avec dropout et batch norm de l'étape 7. On lit les mêmes indicateurs : l'**écart**
entre entraînement et validation, et la **loss de validation**.

Le code :
- pour chaque nom, un CNN neuf, entraîné par `entrainer_avec_scheduler`, puis mesuré ;
- `bilans` garde une ligne de résumé par entraînement, affichées ensemble à la fin.

Prédisez : avec un learning rate qui baisse en fin d'entraînement, les dernières époques seront-elles plus
stables ou moins stables ?

In [ ]:
bilans = []
for nom in ["lr constant", "StepLR", "CosineAnnealingLR"]:
    print(nom)
    torch.manual_seed(0)
    model = CNN(batchnorm=True, dropout=0.25).to(device)
    entrainer_avec_scheduler(model, nom)
    (acc_trn, loss_trn), (acc_val, loss_val) = (
        mesurer(model, train_ds),
        mesurer(model, val_ds),
    )
    bilans.append(
        f"{nom:18s} train_acc {acc_trn:.3f} val_acc {acc_val:.3f}  écart {acc_trn - acc_val:.3f}  train_loss {loss_trn:.3f}  val_loss {loss_val:.3f}"
    )
print()
print("\n".join(bilans))


*À commenter après exécution.*

## 14. Les trois learning rates dans TensorBoard

**Les courbes des trois entraînements, avec le learning rate de chacun.**

Le code : `%tensorboard --logdir runs/scheduler` affiche les courbes de ce dossier (l'extension est déjà
chargée). Le graphique `learning rate` retrouve l'escalier de StepLR et la descente douce du cosinus de
l'étape 11 ; on le compare aux courbes de loss et d'accuracy.

In [ ]:
%tensorboard --logdir runs/scheduler

Dans Colab, le tableau de bord de TensorBoard s'affiche sous la cellule ; ce notebook enregistré n'en garde pas l'image. Le graphique `learning rate` montre les trois réglages ; les courbes de loss et d'accuracy montrent leur effet époque par époque.

## 15. Gradient clipping

**Le gradient clipping borne la taille des gradients.**

Parfois, un lot produit des gradients énormes ; le pas de correction devient démesuré et
l'entraînement déraille. Le clipping réduit les gradients quand leur norme totale dépasse un seuil, en gardant
leur direction.

Le code :
- un `backward()` sur un lot, pour obtenir des gradients ;
- `clip_grad_norm_(model.parameters(), max_norm=0.5)` : renvoie la norme de tous les gradients réunis, puis
  les réduit si elle dépasse 0.5 ;
- on recalcule la norme après, pour vérifier. Dans une boucle, cette ligne se place entre `backward()` et
  `step()`.

In [ ]:
torch.manual_seed(0)
model = CNN().to(device)
images, labels = next(iter(DataLoader(train_ds, batch_size=32, shuffle=True)))
nn.functional.cross_entropy(model(images.to(device)), labels.to(device)).backward()
norme_avant = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
norme_apres = torch.norm(torch.stack([p.grad.norm() for p in model.parameters()]))
print(f"norme avant : {norme_avant.item():.3f}   après : {norme_apres.item():.3f}")

La norme de tous les gradients réunis valait 0.571, au-dessus du seuil `max_norm=0.5` : `clip_grad_norm_` les a réduits jusqu'à une norme de 0.500. La direction du pas est conservée, seule sa longueur est bornée.

**À retenir** : dans la boucle, le clipping se place entre `backward()` et `step()`.

## À retenir de cette démo

- Le **surapprentissage** se lit dans l'écart entre entraînement et validation ; dropout et batch norm le réduisent.
- `model.train()` et `model.eval()` changent le comportement du dropout et de la batch norm : toujours passer en `eval()` pour mesurer.
- Un scheduler fait varier le learning rate au fil des époques ; le clipping borne la taille des pas.